# Exp 2 — Enhanced Augmentation

**Hypothesis:** Adding vertical flip, rotation, colour jitter, affine, and random
erasing reduces overfitting and improves generalisation, especially for minority grades.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

In [ ]:
EXP_DIR  = ROOT / 'outputs' / 'swinv2_small_exp2'
CKPT_DIR_E = EXP_DIR / 'checkpoints'
FIG_DIR_E  = EXP_DIR / 'figures'
for d in (CKPT_DIR_E, FIG_DIR_E): d.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = CKPT_DIR_E / 'exp2_best.pt'

In [ ]:
# ── DataLoaders — enhanced augmentation + WRS ────────────────────────────────
train_ds    = DRDataset(train_df, TRAIN_DIR, transform=enhanced_train_transform)
val_ds      = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
test_ds     = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)
t_loader    = make_loader(train_ds, sampler=wrs_sampler)
v_loader    = make_val_loader(val_ds)
test_loader = make_val_loader(test_ds)
print('Enhanced augmentation: v-flip, rotation, colour jitter, affine, random erasing')

In [ ]:
set_seed()
model     = build_swin_model(pretrained=True).to(device)
criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
optimizer = layer_wise_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

In [ ]:
history, best_kappa = train_loop(
    model, criterion, optimizer, scheduler, scaler,
    t_loader, v_loader, CKPT_PATH)

In [ ]:
y_true, y_pred = evaluate(model, criterion, optimizer, scaler,
                           test_loader, CKPT_PATH, 'Exp2')
plot_confusion(y_true, y_pred, 'Exp2 Enhanced Aug', FIG_DIR_E / 'exp2_confusion.png', cmap='Greens')

import json as _json
results = {'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
           'macro_f1': f1_score(y_true, y_pred, average='macro'),
           'f1_per': f1_score(y_true, y_pred, average=None).tolist(), 'history': history}
with open(EXP_DIR / 'exp2_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print(f"Exp2 QWK: {results['qwk']:.4f}")